# RAGAS Evaluation — MSADS RAG Agent

End-to-end quality evaluation using RAGAS metrics:
- **Faithfulness** — answer supported by context, no hallucination
- **AnswerRelevancy** — answer addresses the question
- **ContextPrecision** — fraction of retrieved context that is relevant
- **ContextRecall** — retrieved context covers the reference answer

LLM: `qwen3:8b` via Ollama  
Embeddings: `BAAI/bge-small-en-v1.5`

## 1. Environment Check

In [3]:
import sys, json, urllib.request
from pathlib import Path

# --- ragas version ---
try:
    import ragas
    print(f"ragas version: {ragas.__version__}")
except ImportError:
    print("ragas not installed — run: pip install 'ragas>=0.2' langchain-ollama langchain-community")

# --- Ollama connectivity ---
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5) as r:
        tags = json.loads(r.read())
    models = [m["name"] for m in tags.get("models", [])]
    print(f"Ollama available. Models: {models}")
except Exception as e:
    print(f"Ollama not available: {e}")

# --- index files ---
INDEX_DIR = Path("../index")
for f in ["chunks.json", "knowledge_graph.json", "bm25.pkl", "index_meta.json"]:
    p = INDEX_DIR / f
    print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

# --- eval_queries.json ---
QUERIES_PATH = Path("../eval_queries.json")
print(f"{'OK' if QUERIES_PATH.exists() else 'MISSING'}: {QUERIES_PATH}")

ragas version: 0.4.3
Ollama available. Models: ['qwen3:8b']
OK: ..\index\chunks.json
OK: ..\index\knowledge_graph.json
OK: ..\index\bm25.pkl
OK: ..\index\index_meta.json
OK: ..\eval_queries.json


## 2. Load eval_queries.json

In [4]:
import json
from pathlib import Path

QUERIES_PATH = Path("../eval_queries.json")
eval_queries = json.loads(QUERIES_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(eval_queries)} evaluation queries.")
print("Sample:", json.dumps(eval_queries[0], indent=2, ensure_ascii=False))

Loaded 30 evaluation queries.
Sample: {
  "qid": "C1-Q01",
  "category": "admission",
  "sub_type": "single_fact",
  "difficulty": "easy",
  "question": "How many letters of recommendation do I need for the MS in Applied Data Science?",
  "ground_truth_answer": "The program requires two letters of recommendation. They strongly recommend that at least one come from a direct manager, supervisor, or internship supervisor who can speak to your professional skills. Letters from family members, friends, or peers are not accepted.",
  "gold_urls": [
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/"
  ],
  "gold_section_hint": "How to Apply — Letters of Recommendation",
  "expected_behavior": "direct_answer",
  "notes": "Core admission fact, low ambiguity."
}


## 3. Run Agent — Collect Answers and Contexts

In [5]:
import sys, json
from pathlib import Path

# Add project root to path so agent/ and grag/ are importable
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)  # work relative to project root

from agent import agent_loop, answer_generator
from agent.ollama_client import OllamaClient
from agent.tools import AgentTools

tools = AgentTools(index_dir="index")
client = OllamaClient()
print(f"Ollama available: {client.is_available()}")

Ollama available: True


### 3a. 手动测试单条 Query（肉眼看效果）

把 `query` 改成你想问的问题，运行这个 cell，直接看 agent 的回答和证据。  
这个 cell 独立运行，不影响后面的批量 eval。

In [9]:
# ====== 改这里 ======
query = "What application materials do I need to submit for the MS in Applied Data Science?"
# ====================

memory = agent_loop.run(query=query, history=[], tools=tools, client=client)
response = answer_generator.generate(memory, client)

# --- 运行摘要 ---
print(f"stop_reason : {memory.stop_reason}")
print(f"tool_calls  : {len(memory.tool_calls)}")
print(f"evidence    : {len(memory.evidence_container)} items")

# --- 改写后的 query ---
print("\nRewritten queries:")
for rq in memory.rewritten_queries:
    print(f"  [{rq.get('id','')}] {rq.get('query','')}")

# --- 每条 tool call ---
print("\nTool calls:")
for tc in memory.tool_calls:
    print(f"  step {tc.step}: {tc.tool_name}  args={tc.arguments}")

# --- 保留的 evidence ---
print("\nEvidence kept:")
for ev in memory.evidence_container:
    path = " > ".join(ev.path or [])
    print(f"  [{ev.evidence_id}] {ev.page_title}")
    print(f"           path : {path}")
    print(f"           url  : {ev.source_url}")
    print(f"           why  : {ev.why_kept}")
    print()

# --- 最终回答 ---
print("=" * 60)
print("ANSWER")
print("=" * 60)
print(response.answer)
print()
print("CITATIONS")
for c in response.citations:
    print(f"  [{c.index}] {c.title}")
    print(f"       {c.source_url}")
    print(f"       snippet: {c.snippet[:120]}")

stop_reason : sufficient_evidence
tool_calls  : 4
evidence    : 6 items

Rewritten queries:
  [q1] What are the required application materials for the MS in Applied Data Science?

Tool calls:
  step 1: hybrid_retrieve  args={'query': 'What are the required application materials for the MS in Applied Data Science?', 'top_k': 8}
  step 2: list_page_summaries  args={'query': 'required application materials MSADS', 'limit': 5}
  step 3: inspect_page  args={'page_id_or_label_or_title': 'How to Apply | page:7fb2f6ecae8feeeb'}
  step 4: fetch_node_chunks  args={'node_id': 'page:7fb2f6ecae8feeeb', 'fetch_depth': 1}

Evidence kept:
  [ev_001] How to Apply
           path : How to Apply > Letters of Recommendation
           url  : https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply
           why  : judge kept (step 4)

  [ev_002] How to Apply
           path : How to Apply > Candidate Statement
           url  : https://datascience.uchicago.edu/e

### 3b. 批量运行所有 eval_queries（用于 RAGAS）

In [9]:
from tqdm.auto import tqdm

CACHE_PATH = Path("agent_run_cache.json")

# Load cache if exists
if CACHE_PATH.exists():
    cache = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cache with {len(cache)} entries.")
else:
    cache = {}

results = []

for row in tqdm(eval_queries, desc="Running agent"):
    qid = row["qid"]
    question = row["question"]
    ground_truth = row.get("ground_truth_answer", "")

    if qid in cache:
        results.append(cache[qid])
        continue

    # Run agent
    memory = agent_loop.run(
        query=question,
        history=[],
        tools=tools,
        client=client,
    )
    response = answer_generator.generate(memory, client)

    contexts = [ev.text for ev in memory.evidence_container]

    entry = {
        "qid": qid,
        "question": question,
        "agent_answer": response.answer,
        "contexts": contexts,
        "ground_truth_answer": ground_truth,
        "stop_reason": memory.stop_reason,
        "tool_call_count": len(memory.tool_calls),
    }
    cache[qid] = entry
    results.append(entry)

    # Save cache incrementally
    CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nCollected {len(results)} results.")

Loaded cache with 30 entries.


Running agent:   0%|          | 0/30 [00:00<?, ?it/s]


Collected 30 results.


## 4. Build RAGAS EvaluationDataset

In [10]:
from ragas import EvaluationDataset, SingleTurnSample

samples = [
    SingleTurnSample(
        user_input=row["question"],
        response=row["agent_answer"],
        retrieved_contexts=row["contexts"],
        reference=row["ground_truth_answer"],
    )
    for row in results
    if row["contexts"]  # skip empty-evidence runs for metric stability
]

dataset = EvaluationDataset(samples=samples)
print(f"Dataset: {len(samples)} samples")

Dataset: 25 samples


## 5. Configure RAGAS LLM + Embeddings

In [11]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

# qwen3:8b thinking mode may interfere with RAGAS JSON parsing
# Use /no_think in system prompt or stop token to disable
ragas_llm = LangchainLLMWrapper(
    ChatOllama(
        model="qwen3:8b",
        base_url="http://localhost:11434",
        # Disable thinking mode for reliable RAGAS parsing
        model_kwargs={"options": {"stop": ["</think>"]}},
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)

print("RAGAS LLM and embeddings configured.")

C:\Users\ss363\AppData\Local\Temp\ipykernel_37992\2423663321.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(


RAGAS LLM and embeddings configured.


C:\Users\ss363\AppData\Local\Temp\ipykernel_37992\2423663321.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


## 6. Run RAGAS evaluate()

Estimated: ~30 questions × 4 metrics × ~2 LLM calls ≈ 240 Ollama calls (~20–40 min locally)

In [13]:
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.run_config import RunConfig

result = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall(),
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=RunConfig(max_workers=1, timeout=500),
)

df = result.to_pandas()
print(df.head())

C:\Users\ss363\AppData\Local\Temp\ipykernel_37992\918643368.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\ss363\AppData\Local\Temp\ipykernel_37992\918643368.py:2: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
C:\Users\ss363\AppData\Local\Temp\ipykernel_37992\918643368.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metri

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

Exception raised in Job[39]: OutputParserException(Failed to parse StringIO from completion {"classifications": [{"statement": "The 1-year 12-course program requires 12 courses (6 Core, 4 Electives, 2 Capstone) and is available in both Full- and Part-Time formats; full-time students typically graduate in 4-5 quarters and part-time students may take up to 6 quarters.", "reason": "The program's structure and graduation timelines are explicitly stated in the provided text.", "attributed": true}, {"statement": "The 2-Year Thesis Track is Full-Time only, completed over 6 academic quarters (about 21 months), and requires 18 total courses (6 Core, 8 Electives, 2 Capstone, 2 Independent Study) plus a Master's Thesis, which can be a traditional written thesis or a project-based thesis.", "reason": "The 2-year program's requirements and thesis options are detailed in the text.", "attributed": true}, {"statement": "The 2-year track is intended for students who want a longer program with more elec

                                          user_input  \
0  How many letters of recommendation do I need f...   
1              Is the GRE or GMAT required to apply?   
2                   How much is the application fee?   
3  What application materials do I need to submit...   
4  I'm an international student who needs visa sp...   

                                  retrieved_contexts  \
0  [The MS in Applied Data Science program requir...   
1  [No, the GRE/GMAT is not required for admissio...   
2  [There is a $90 non-refundable application fee...   
3  [# Page Structure: How to Apply\nURL: https://...   
4  [- Only the full-time, in-person program is vi...   

                                            response  \
0  The MS in Applied Data Science program require...   
1  The GRE or GMAT is not required for the Master...   
2  The application fee for the UChicago MS in App...   
3  To apply for the MS in Applied Data Science at...   
4  As an international student requiring visa 

## 7. Print Metric Means

In [14]:
metrics_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
available = [c for c in metrics_cols if c in df.columns]
print("\n=== RAGAS Metric Means ===")
print(df[available].mean().to_string())


=== RAGAS Metric Means ===
faithfulness         0.746861
answer_relevancy     0.735263
context_precision    0.652278
context_recall       0.525362


## 8. Low-Score Sample Analysis

In [15]:
import pandas as pd

for metric in available:
    print(f"\n=== Lowest 5 by {metric} ===")
    low = df.nsmallest(5, metric)[["user_input", "response", metric]]
    for _, row in low.iterrows():
        print(f"  Score={row[metric]:.3f}")
        print(f"  Q: {str(row['user_input'])[:120]}")
        print(f"  A: {str(row['response'])[:200]}")
        print()


=== Lowest 5 by faithfulness ===
  Score=0.000
  Q: What courses does Arnab Bose teach in the program?
  A: Arnab Bose teaches the following courses in the UChicago MS in Applied Data Science program: Machine Learning, Machine Learning Operations, Time Series Analysis and Forecasting, and Data Science in He

  Score=0.250
  Q: Who is the director of the MS in Applied Data Science program at UChicago?
  A: The director of the MS in Applied Data Science program at UChicago is David Uminsky, PhD, who is listed as the Executive Director of the UChicago Data Science Institute [1]. The evidence provided does

  Score=0.333
  Q: How much is the application fee?
  A: The application fee for the UChicago MS in Applied Data Science program is $90, which is non-refundable. For questions regarding an application fee waiver, please refer to the Physical Sciences Divisi

  Score=0.400
  Q: What does the online MS-ADS student experience look like week-to-week?
  A: The online MS in Applied Data Scie

## 9. Export Results

In [16]:
OUT_PATH = Path("ragas_results.csv")
df.to_csv(OUT_PATH, index=False)
print(f"Results saved to {OUT_PATH.resolve()}")

Results saved to C:\Users\ss363\Desktop\spring2026\Gen AI\RAG-based-Interactive-AI-for-MSADS-Website\ragas_results.csv
